# Fuzzy matching for SEO: an end-to-end notebook

Companion code for the MLforSEO post **What Fuzzy Matching Is, and Where It Fits in SEO** (merged with Lazarina Stoy's *Ultimate End-to-End Guide to Fuzzy Matching for SEOs*).

This notebook recreates every tab of the free Google Sheets template in Python, then compares the main libraries side by side:

1. **Scorers 101**: what `ratio`, `partial_ratio`, `token_sort_ratio`, `token_set_ratio` and `WRatio` actually measure (and the HARD / HAND / HARDER example).
2. **String assessment and comparisons**: for each keyword or title, find its nearest neighbour in the same list (cannibalisation and duplicate checks).
3. **URL redirect mapping**: map old URLs to the most similar new URL, label direct matches vs mismatches, bucket by confidence.
4. **Internal linking opportunities**: compare short content extracts between pages and flag links that don't exist yet.
5. **Library comparison**: RapidFuzz vs TheFuzz (the renamed FuzzyWuzzy) vs PolyFuzz vs fuzzy-pandas.
6. **Phonetic matching** with Metaphone and NYSIIS.
7. **Why fuzzy matching is not a keyword-clustering tool** (and the semantic upgrade path).

Everything runs on the built-in sample data. Swap in your own CSVs where marked `# YOUR DATA`.

*Tested October 2026 with RapidFuzz 3.13–3.14, TheFuzz 0.22.1, PolyFuzz 0.4.3, pandas 2.3.*

In [1]:
# Colab already has pandas and scikit-learn. Install the fuzzy-matching libraries.
%pip install -q "rapidfuzz>=3.9" "thefuzz>=0.22" "polyfuzz>=0.4.3" "jellyfish>=0.9"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from rapidfuzz import fuzz, process, distance, utils

pd.set_option("display.max_colwidth", 80)
import rapidfuzz, polyfuzz
print("RapidFuzz", rapidfuzz.__version__, "| PolyFuzz", polyfuzz.__version__)

RapidFuzz 3.13.0 | PolyFuzz 0.4.3


## 1. Scorers 101

RapidFuzz (and TheFuzz, which now runs on RapidFuzz under the hood) gives you a family of scorers that all return 0–100:

| Scorer | What it does | Use it for |
|---|---|---|
| `fuzz.ratio` | Normalised Indel similarity (insertions + deletions) of the whole strings | Short strings of similar length |
| `fuzz.partial_ratio` | Best `ratio` of the shorter string against any same-length window of the longer one | "Is this keyword inside that title?" |
| `fuzz.token_sort_ratio` | Lower-cases, splits into words, sorts them, then `ratio` | Word-order changes ("api google trends" vs "google trends api") |
| `fuzz.token_set_ratio` | Compares the shared words vs the leftovers | One string is a superset of the other |
| `fuzz.WRatio` | A weighted blend of the above | A sensible default when you don't know yet |

And for the raw edit distances: `distance.Levenshtein`, `distance.DamerauLevenshtein` (counts transpositions), `distance.JaroWinkler`, `distance.Indel` and more.

In [3]:
pairs = [("hard", "hand"), ("hard", "harder")]
rows = []
for a, b in pairs:
    rows.append({
        "pair": f"{a} / {b}",
        "Levenshtein distance (edits)": distance.Levenshtein.distance(a, b),
        "fuzz.ratio (0-100)": round(fuzz.ratio(a, b), 1),
        "Jaro-Winkler (0-1)": round(distance.JaroWinkler.normalized_similarity(a, b), 3),
    })
pd.DataFrame(rows)

,pair,Levenshtein distance (edits),fuzz.ratio (0-100),Jaro-Winkler (0-1)
0,hard / hand,1,75.0,0.867
1,hard / harder,2,80.0,0.933


**Read this table carefully.** Raw Levenshtein says *hard/hand* (1 edit) is closer than *hard/harder* (2 edits), which is the classic "no sense of meaning" example. But the normalised `fuzz.ratio` that most SEO scripts actually use scores *hard/harder* higher (80 vs 75), because it divides by the combined length and doesn't count substitutions as one edit. Neither of them understands meaning; they just disagree about spelling. Always check which score your library is giving you.

In [4]:
examples = [
    ("google trends api", "api google trends"),
    ("fuzzy matching", "ultimate guide to fuzzy matching for seo"),
    ("/blog/fuzy-matching/", "/blog/fuzzy-matching-guide/"),
]
pd.DataFrame([{
    "a": a, "b": b,
    "ratio": round(fuzz.ratio(a, b)),
    "partial_ratio": round(fuzz.partial_ratio(a, b)),
    "token_sort_ratio": round(fuzz.token_sort_ratio(a, b)),
    "token_set_ratio": round(fuzz.token_set_ratio(a, b)),
    "WRatio": round(fuzz.WRatio(a, b)),
} for a, b in examples])

,a,b,ratio,partial_ratio,token_sort_ratio,token_set_ratio,WRatio
0,google trends api,api google trends,76,87,100,100,95
1,fuzzy matching,ultimate guide to fuzzy matching for seo,52,100,52,100,90
2,/blog/fuzy-matching/,/blog/fuzzy-matching-guide/,85,95,85,85,85


## 2. String assessment and comparisons (Sheets tab 1)

The Sheets template takes one column of keywords, titles or headings and, for each one, returns the most similar *other* string in the list plus a similarity score. High scores flag near-duplicates and possible cannibalisation; zero means nothing passed the threshold.

In [5]:
# YOUR DATA: replace with a column of keywords, titles or H1s
strings = [
    "google trends api", "pytrends", "google trend api", "lazarina stoy",
    "google trends api python", "google trends python", "api google trends",
    "pytrend", "python google trends", "google data studio limitations",
    "pytrends example", "data studio limitations", "lda topic modeling",
    "lda topic modelling", "pytrends documentation", "auto generate meta description",
]

def nearest_in_same_list(items, scorer=fuzz.token_sort_ratio, threshold=30):
    # Score every string against every other string in one vectorised call
    m = process.cdist(items, items, scorer=scorer, processor=utils.default_process, workers=-1)
    out = []
    for i, s in enumerate(items):
        m[i, i] = -1  # ignore self-match
        j = int(m[i].argmax())
        score = float(m[i, j])
        out.append({
            "string": s,
            "most_similar": items[j] if score >= threshold else "",
            "similarity": round(score, 1) if score >= threshold else 0.0,
        })
    return pd.DataFrame(out).sort_values("similarity", ascending=False)

assessment = nearest_in_same_list(strings)
assessment

,string,most_similar,similarity
0,google trends api,api google trends,100.0
5,google trends python,python google trends,100.0
6,api google trends,google trends api,100.0
8,python google trends,google trends python,100.0
12,lda topic modeling,lda topic modelling,97.3
13,lda topic modelling,lda topic modeling,97.3
2,google trend api,google trends api,97.0
1,pytrends,pytrend,93.3
7,pytrend,pytrends,93.3
4,google trends api python,google trends python,90.9


## 3. URL redirect mapping (Sheets tab 2)

Old URLs in one list, new URLs in another. For each old URL: is there an exact match? If not, what's the closest new URL and how confident are we? Matching on the **path only** (not the domain) and stripping punctuation gives much better results than comparing full URLs.

For a fuller, production-ready version with a crawl-based workflow, see the MLforSEO tutorial *404 and redirect mapping with fuzzy matching in Google Colab*.

In [6]:
from urllib.parse import urlparse
import re

# YOUR DATA: old URLs (pre-migration crawl, GSC, backlink exports) and new URLs (fresh crawl)
old_urls = [
    "https://example.com/blog/ultimate-guide-to-pytrends/",
    "https://example.com/blog/data-studio-limitations/",
    "https://example.com/blog/topic-modelling-lda/",
    "https://example.com/blog/sentiment-analysis-theory/",
    "https://example.com/about-me/",
    "https://example.com/blog/fuzzy-string-matching-google-sheets-template/",
    "https://example.com/old-promo-page-2021/",
]
new_urls = [
    "https://example.com/guides/pytrends-google-trends-python/",
    "https://example.com/looker-studio/data-studio-limitations/",
    "https://example.com/guides/topic-modeling-with-lda/",
    "https://example.com/guides/sentiment-analysis/",
    "https://example.com/about/",
    "https://example.com/guides/fuzzy-matching-for-seo/",
]

def path_key(url):
    p = urlparse(url).path.lower().strip("/")
    return re.sub(r"[-_/]+", " ", p)

def map_redirects(old, new, scorer=fuzz.token_set_ratio, high=85, low=60):
    new_keys = [path_key(u) for u in new]
    rows = []
    for u in old:
        if u in new:
            rows.append({"old_url": u, "match": "direct match", "new_url": u, "similarity": 100.0})
            continue
        best = process.extractOne(path_key(u), new_keys, scorer=scorer)
        key, score, idx = best
        rows.append({"old_url": u, "match": "fuzzy match", "new_url": new[idx], "similarity": round(score, 1)})
    df = pd.DataFrame(rows)
    df["action"] = pd.cut(df["similarity"], bins=[-1, low, high, 101],
                          labels=["no good match: review / category page / let it 404",
                                  "partial match: review by hand",
                                  "confident: redirect"])
    return df.sort_values("similarity", ascending=False)

redirects = map_redirects(old_urls, new_urls)
redirects

,old_url,match,new_url,similarity,action
4,https://example.com/about-me/,fuzzy match,https://example.com/about/,100.0,confident: redirect
1,https://example.com/blog/data-studio-limitations/,fuzzy match,https://example.com/looker-studio/data-studio-limitations/,90.2,confident: redirect
3,https://example.com/blog/sentiment-analysis-theory/,fuzzy match,https://example.com/guides/sentiment-analysis/,83.7,partial match: review by hand
2,https://example.com/blog/topic-modelling-lda/,fuzzy match,https://example.com/guides/topic-modeling-with-lda/,74.1,partial match: review by hand
0,https://example.com/blog/ultimate-guide-to-pytrends/,fuzzy match,https://example.com/guides/pytrends-google-trends-python/,68.7,partial match: review by hand
5,https://example.com/blog/fuzzy-string-matching-google-sheets-template/,fuzzy match,https://example.com/guides/fuzzy-matching-for-seo/,65.1,partial match: review by hand
6,https://example.com/old-promo-page-2021/,fuzzy match,https://example.com/guides/fuzzy-matching-for-seo/,29.2,no good match: review / category page / let it 404


Tune `high` and `low` on a sample you've checked by hand. Notice `/about-me/` scores 100 against `/about/`: `token_set_ratio` gives full marks when one string's words are a subset of the other's. That's right here, but it's also how `/blog/` would "perfectly" match `/blog/anything/`, so look at short paths with extra care. A token-based scorer copes with re-ordered slugs; plain `ratio` is stricter. Remember: this compares URL **strings**, not page content. If the content changed between the old and new page, a high score can still be the wrong redirect.

## 4. Internal linking opportunities (Sheets tab 3)

The Sheets version takes a Screaming Frog custom extraction of body content plus each page's URL, finds the most similar other page, and checks your inlinks report to see whether a link already exists.

Two caveats from the original guide still apply:
- only compare pages that are similar in **structure** (product pages with product pages), otherwise you compare apples to oranges;
- keep the strings short (titles, H1s, intro paragraphs rather than whole pages) and **sense-check every suggestion**. Fuzzy matching sees shared characters, not shared meaning, so this is a shortlist generator, not a decision-maker. For meaning-based link suggestions, use embeddings (section 7).

In [7]:
# YOUR DATA: a short content extract per page (title + intro works well) and the page URL
pages = pd.DataFrame({
    "url": ["/fuzzy-matching-seo/", "/redirect-mapping-python/", "/topic-modeling-lda/",
            "/topic-modeling-bertopic/", "/entity-extraction-seo/", "/404-redirect-mapping-fuzzy/"],
    "extract": [
        "fuzzy matching for seo what it is how string similarity works",
        "redirect mapping in python with fuzzy matching for site migrations",
        "topic modeling for seo with lda latent dirichlet allocation",
        "topic modeling for seo with bertopic embeddings and clustering",
        "entity extraction for seo with named entity recognition",
        "404 and redirect mapping with fuzzy matching in google colab",
    ],
})
# YOUR DATA: Screaming Frog 'All Inlinks' export reduced to Source/Destination
existing_links = pd.DataFrame({"Source": ["/redirect-mapping-python/"], "Destination": ["/404-redirect-mapping-fuzzy/"]})

def internal_link_opportunities(df, scorer=fuzz.token_set_ratio, min_score=50):
    m = process.cdist(df["extract"].tolist(), df["extract"].tolist(), scorer=scorer, workers=-1)
    linkset = set(zip(existing_links["Source"], existing_links["Destination"]))
    rows = []
    for i in range(len(df)):
        m[i, i] = -1
        j = int(m[i].argmax())
        score = float(m[i, j])
        rows.append({
            "page": df.url[i],
            "most_similar_page": df.url[j],
            "similarity": round(score, 1),
            "verdict": "possible duplicate: check" if score >= 95 else
                       ("review this opportunity" if score >= min_score else "no strong candidate"),
            "link_already_exists": (df.url[i], df.url[j]) in linkset,
        })
    return pd.DataFrame(rows).sort_values("similarity", ascending=False)

internal_link_opportunities(pages)

,page,most_similar_page,similarity,verdict,link_already_exists
1,/redirect-mapping-python/,/404-redirect-mapping-fuzzy/,78.8,review this opportunity,True
5,/404-redirect-mapping-fuzzy/,/redirect-mapping-python/,78.8,review this opportunity,False
2,/topic-modeling-lda/,/topic-modeling-bertopic/,67.8,review this opportunity,False
3,/topic-modeling-bertopic/,/topic-modeling-lda/,67.8,review this opportunity,False
0,/fuzzy-matching-seo/,/redirect-mapping-python/,56.7,review this opportunity,False
4,/entity-extraction-seo/,/topic-modeling-bertopic/,54.5,review this opportunity,False


## 5. Library comparison: RapidFuzz, TheFuzz (FuzzyWuzzy), PolyFuzz, fuzzy-pandas

**Status in October 2026:**
- **FuzzyWuzzy** was renamed **TheFuzz** in 2021 and the old `seatgeek/fuzzywuzzy` repo is archived. TheFuzz keeps the same API but now uses **RapidFuzz** as its engine, so `from thefuzz import fuzz` gives you RapidFuzz scores.
- **RapidFuzz** is the actively maintained, MIT-licensed, C++-backed library most people should start with. Same scorers as FuzzyWuzzy, plus `process.cdist` for whole-matrix scoring and many distance metrics.
- **PolyFuzz** (Maarten Grootendorst) is still maintained (v0.4.3, 2025) and is the best *framework* for comparing models: RapidFuzz, edit distance, TF-IDF n-grams, and embedding models (Sentence-Transformers, Flair, spaCy, Gensim) in one object, with grouping and precision-recall plots.
- **fuzzy-pandas** (last release 2019, wraps `csvmatch`) still installs, but it's unmaintained. Use pandas + RapidFuzz instead (shown below).

In [8]:
from thefuzz import fuzz as thefuzz_fuzz
a, b = "google trends api python", "python google trends api"
print("RapidFuzz token_sort_ratio:", fuzz.token_sort_ratio(a, b))
print("TheFuzz   token_sort_ratio:", thefuzz_fuzz.token_sort_ratio(a, b))

RapidFuzz token_sort_ratio: 100.0
TheFuzz   token_sort_ratio: 100


In [9]:
from polyfuzz import PolyFuzz
from polyfuzz.models import RapidFuzz, TFIDF, EditDistance
from rapidfuzz.distance import JaroWinkler

from_list = ["apple", "apples", "appl", "recal", "house", "similarity", "pytrend", "lda topic modelling"]
to_list   = ["apple", "apples", "mouse", "pytrends", "lda topic modeling"]

models = [
    RapidFuzz(n_jobs=1, scorer=fuzz.WRatio, model_id="RapidFuzz WRatio"),
    TFIDF(n_gram_range=(3, 3), min_similarity=0, model_id="TF-IDF char 3-grams"),
    EditDistance(n_jobs=1, scorer=JaroWinkler.normalized_similarity, model_id="Jaro-Winkler"),
]
pf = PolyFuzz(models).match(from_list, to_list)

# Side-by-side view of what each model picked
comparison = pd.DataFrame({"From": from_list})
for mid in pf.get_ids():
    m = pf.get_matches(mid)
    comparison[f"{mid} -> To"] = m["To"].values
    comparison[f"{mid} score"] = m["Similarity"].round(2).values
comparison

,From,RapidFuzz WRatio -> To,RapidFuzz WRatio score,TF-IDF char 3-grams -> To,TF-IDF char 3-grams score,Jaro-Winkler -> To,Jaro-Winkler score
0,apple,apple,1.00,apple,1.00,apple,1.00
1,apples,apples,1.00,apples,1.00,apples,1.00
2,appl,apples,0.90,apple,0.79,apple,0.92
3,recal,apple,0.40,None,0.00,pytrends,0.14
4,house,mouse,0.80,mouse,0.60,mouse,0.74
5,similarity,lda topic modeling,0.36,None,0.00,lda topic modeling,0.00
6,pytrend,pytrends,0.93,pytrends,0.89,pytrends,0.95
7,lda topic modelling,lda topic modeling,0.97,lda topic modeling,0.82,lda topic modeling,0.98


In [10]:
# PolyFuzz can also group the matched "To" strings into clusters.
# The model argument is the model used for grouping (a model object, "TF-IDF", "EditDistance" or "Embeddings").
pf.group(model=TFIDF(n_gram_range=(3, 3)), link_min_similarity=0.75)
pf.get_matches("TF-IDF char 3-grams")

,From,To,Similarity,Group
0,apple,apple,1.000,apples
1,apples,apples,1.000,apples
2,appl,apple,0.790,apples
3,recal,None,0.000,None
4,house,mouse,0.598,mouse
5,similarity,None,0.000,None
6,pytrend,pytrends,0.888,pytrends
7,lda topic modelling,lda topic modeling,0.818,lda topic modeling


### fuzzy-pandas replacement: a fuzzy merge in plain pandas + RapidFuzz

In [11]:
left  = pd.DataFrame({"title": ["Fuzzy matching guide", "Topic modelling with LDA", "Sentiment analysis basics"]})
right = pd.DataFrame({"title": ["fuzzy matching guid", "LDA topic modeling", "Entity extraction 101"],
                      "clicks": [120, 80, 45]})

def fuzzy_merge(left, right, left_on, right_on, scorer=fuzz.token_sort_ratio, threshold=70):
    choices = right[right_on].tolist()
    matches = [process.extractOne(x, choices, scorer=scorer, processor=utils.default_process) for x in left[left_on]]
    out = left.copy()
    out["match"] = [m[0] if m[1] >= threshold else None for m in matches]
    out["score"] = [round(m[1], 1) for m in matches]
    return out.merge(right, how="left", left_on="match", right_on=right_on, suffixes=("", "_right"))

fuzzy_merge(left, right, "title", "title")

,title,match,score,title_right,clicks
0,Fuzzy matching guide,fuzzy matching guid,97.4,fuzzy matching guid,120.0
1,Topic modelling with LDA,LDA topic modeling,85.7,LDA topic modeling,80.0
2,Sentiment analysis basics,None,34.8,NaN,NaN


## 6. Phonetic matching: Metaphone and NYSIIS

Phonetic algorithms encode how a word *sounds*, so misspelled brand names often collapse to the same code. Low precision (lots of false positives), so use them as a pre-filter or alongside an edit-distance score.

In [12]:
import jellyfish
names = ["Lazarina", "Lazareena", "Lazarena", "Stoy", "Stoi", "MLforSEO", "ML for SEO"]
pd.DataFrame({"name": names,
              "metaphone": [jellyfish.metaphone(n) for n in names],
              "nysiis": [jellyfish.nysiis(n) for n in names]})

,name,metaphone,nysiis
0,Lazarina,LSRN,LASARAN
1,Lazareena,LSRN,LASARAN
2,Lazarena,LSRN,LASARAN
3,Stoy,ST,STY
4,Stoi,ST,ST
5,MLforSEO,MLFRS,MLFARS
6,ML for SEO,ML FR S,ML


## 7. Why fuzzy matching is not a keyword-clustering tool

It *can* group keywords, and PolyFuzz will happily do it. The problem is that letters, not meaning, decide the groups.

In [13]:
keywords = ["seo audit", "seo audit tool", "site audit", "technical seo checklist",
            "audio editing", "auto seo", "link building", "backlink outreach"]
kpf = PolyFuzz("TF-IDF").match(keywords)
kpf.group(link_min_similarity=0.4)
kpf.get_matches()[["From", "To", "Similarity", "Group"]]

,From,To,Similarity,Group
0,seo audit,seo audit tool,0.668,seo audit
1,seo audit tool,seo audit,0.668,seo audit
2,site audit,seo audit,0.531,seo audit
3,technical seo checklist,backlink outreach,0.115,backlink outreach
4,audio editing,seo audit,0.391,seo audit
5,auto seo,seo audit,0.205,seo audit
6,link building,backlink outreach,0.160,backlink outreach
7,backlink outreach,link building,0.160,link building


Look at the scores: "audio editing" scores about 0.39 against "seo audit" and lands in its group (shared letters: *aud*, *it*), while "link building" and "backlink outreach", which mean nearly the same thing, score only about 0.16. Letters decide, meaning doesn't. For meaning-based grouping, swap in an embedding model. PolyFuzz supports this directly; set `RUN_SEMANTIC = True` to try it (downloads a ~90MB model).

In [14]:
RUN_SEMANTIC = False

if RUN_SEMANTIC:
    %pip install -q sentence-transformers
    from polyfuzz.models import SentenceEmbeddings
    sem = SentenceEmbeddings("sentence-transformers/all-MiniLM-L6-v2", min_similarity=0)
    spf = PolyFuzz(sem).match(keywords)
    display(spf.get_matches())
else:
    print("Semantic comparison skipped. Set RUN_SEMANTIC = True to run it.")

Semantic comparison skipped. Set RUN_SEMANTIC = True to run it.


## 8. Export

In [15]:
assessment.to_csv("string_assessment.csv", index=False)
redirects.to_csv("redirect_map.csv", index=False)
print("Saved string_assessment.csv and redirect_map.csv")
# In Colab: from google.colab import files; files.download("redirect_map.csv")

Saved string_assessment.csv and redirect_map.csv
